In [ ]:
# Papermill will overwrite these at runtime
question = "I need an easy UCORE arts class"
student_context = {}
use_rag = True
top_k = 5

In [ ]:
import os
import json
import time
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

start_time = time.time()

index_path = os.path.join(base_dir, "prompt-search", "data", "domain", "courses.faiss")
meta_path = os.path.join(base_dir, "prompt-search", "data", "domain", "metadata.json")

def semantic_search(query, k):
    try:
        # Requirement: Load cached embeddings/index
        index = faiss.read_index(index_path)
        encoder = SentenceTransformer('all-MiniLM-L6-v2')
        with open(meta_path, 'r', encoding='utf-8') as f:
            metadata = json.load(f)

        # Generate embedding for the search query
        query_vector = encoder.encode([query])
        distances, indices = index.search(np.array(query_vector).astype('float32'), k)
        
        results = []
        sources = []
        
        for i in indices[0]:
            if i == -1: continue
            chunk = metadata[i]
            # Handle both dictionary and list formats for metadata
            course_code = chunk.get('course_code', 'Unknown') if isinstance(chunk, dict) else "Unknown"
            text = chunk.get('chunk_text', '') if isinstance(chunk, dict) else str(chunk)
            
            results.append(f"**{course_code}**\n{text[:200]}...")
            sources.append(course_code)
            
        answer = "Based on your interests, here are the most relevant courses:\n\n" + "\n\n".join(results)
        return answer, sources
    
    except Exception as e:
        return f"Error during semantic search: {str(e)}", []

# Execute
answer, sources = semantic_search(question, top_k)

# Requirement: Output format must match specific JSON schema
output = {
    "answer": answer,
    "sources": sources,
    "metadata": {
        "used_rag": use_rag,
        "model": "all-MiniLM-L6-v2 + FAISS",
        "latency_seconds": round(time.time() - start_time, 2)
    }
}

# Requirement: Write to /tmp/llm_response.json
temp_dir = os.environ.get('TEMP', '/tmp')
output_path = os.path.join(temp_dir, 'llm_response.json')

with open(output_path, 'w') as f:
    json.dump(output, f)

# Requirement: Fallback print to stdout
print(json.dumps(output))